In [ ]:
from include.Python.TELTools import getAzureSecret, getSnowFlakeConn

import requests
import json
import urllib3
import pandas as pd
from pathlib import Path

import yaml
class IndentSafeDumper(yaml.SafeDumper):
    def increase_indent(self, flow=False, indentless=False):
        return super().increase_indent(flow, False)

In [ ]:
headers = {'Content-Type': 'application/json', 'Accept': 'application/json'}

base_url = 'https://telsnus.service-now.com/api/now/table/'
username = getAzureSecret('SERVICE-NOW-API-USER').value
password = getAzureSecret('SERVICE-NOW-API-PASSWORD').value

# base_url = 'https://telsnusdev.service-now.com/api/now/table/'
# username = getAzureSecret('SERVICE-NOW-API-USER-DEV').value
# password = getAzureSecret('SERVICE-NOW-API-PASSWORD-DEV').value

auth = requests.auth.HTTPBasicAuth(username, password)


In [ ]:
def get_table_data(table, params={}):
    response = requests.get(base_url + table, auth=auth, headers=headers, params=params, verify=False)
    status_code = response.status_code
    response.raise_for_status()
    data = response.json()['result']
    print(f'{table=}, {status_code = }, {len(data) = }')
    return data


In [ ]:
params = {
    'sysparm_limit': 10, 
    # 'sysparm_query': 'sys_id=dc7b8a7d93a98f50b130b8e86cba1041^sys_updated_on>=2023-01-01', 
    # 'sysparm_query': 'id=909e39b593698f50b130b8e86cba108b'#^definition.table=incident'#^definition.name=Incident State Duration', 
    'sysparm_query': 'table=sc_task^', 
    # 'sysparm_fields': 'id,sys_id,definition.name',
}
# tbl_keys = {table: get_table_data(table, params)[0].keys() for table in tables}
data = get_table_data('metric_definition', params)
print(json.dumps(data, indent=4))



In [ ]:
tables = (
    'alm_asset',
    'cab_meeting',
    'change_request',
    'cmdb_ci',
    'cmn_department',
    'cmn_location',
    'contract_sla',
    'core_company',
    'incident',
    'interaction',
    'label',
    'label_entry',
    'metric_definition', 
    'metric_instance',
    'sc_cat_item',
    'sc_request',
    'sc_req_item',
    'sc_task',
    'std_change_record_producer',
    'sys_choice',
    'sys_db_object',
    'sys_dictionary',
    'sys_documentation',
    'sys_ui_form',
    'sys_ui_section',
    'sys_user',
    'sys_user_group',
    'task_sla',
)


In [ ]:
tbl_keys = {tbl: list(get_table_data(tbl, {'sysparm_limit': 1})[0].keys()) for tbl in tables}
tbl_keys

In [ ]:
ancestors_sql = f"""
-- For every table, walk UP the inheritance chain and collect all columns
-- defined on the table itself or any ancestor. Shows the full effective
-- column set as it would appear in the API output.
--
-- Result columns:
--   TABLE_NAME         – the table whose effective columns we're listing
--   TABLE_LABEL        – display name of the table from sys_documentation
--   TABLE_DESCRIPTION  – hint/description of the table from sys_documentation
--   COLUMN_NAME        – the field/element name
--   COLUMN_LABEL       – display name of the column from sys_documentation
--   COLUMN_DESCRIPTION – hint/description of the column from sys_documentation
--   DEFINED_ON         – the ancestor table where this column is actually defined
--   INTERNAL_TYPE      – ServiceNow internal type from sys_dictionary
--   INHERITANCE_DEPTH  – 0 = defined on TABLE_NAME itself, 1 = parent, 2 = grandparent, etc.
with recursive ancestors as (
    -- base case: every table is its own ancestor at depth 0
    select 
        dbo.NAME as TABLE_NAME,
        dbo.NAME as ANCESTOR_TABLE,
        dbo.SUPER_CLASS,
        0 as INHERITANCE_DEPTH
    from BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DB_OBJECT as dbo

    union all

    -- recursive step: walk up to the parent
    select 
        a.TABLE_NAME,
        parent.NAME as ANCESTOR_TABLE,
        parent.SUPER_CLASS,
        a.INHERITANCE_DEPTH + 1
    from ancestors as a
    inner join BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DB_OBJECT as child
        on a.ANCESTOR_TABLE = child.NAME
    inner join BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DB_OBJECT as parent
        on child.SUPER_CLASS = parent.SYS_ID
    where child.SUPER_CLASS is not null
)
select 
    a.TABLE_NAME,
    coalesce(table_doc.HINT, table_doc.LABEL) as TABLE_DESCRIPTION,
    dict.ELEMENT as COLUMN_NAME,
    coalesce(col_doc.HINT, col_doc.LABEL) as COLUMN_DESCRIPTION,
    a.ANCESTOR_TABLE as DEFINED_ON,
    dict.INTERNAL_TYPE,
    a.INHERITANCE_DEPTH
from ancestors as a
inner join BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DICTIONARY as dict
    on a.ANCESTOR_TABLE = dict.NAME
left join BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DOCUMENTATION as table_doc
    on a.TABLE_NAME = table_doc.NAME
    and (table_doc.ELEMENT is null or table_doc.ELEMENT = '')
    and table_doc.LANGUAGE = 'en'
left join BRONZE.TEH_SUPPORT_SERVICES.SERVICENOW_SYS_DOCUMENTATION as col_doc
    on dict.NAME = col_doc.NAME
    and dict.ELEMENT = col_doc.ELEMENT
    and col_doc.LANGUAGE = 'en'
where dict.ELEMENT is not null        -- exclude table-level metadata rows
    and dict.ELEMENT != ''
    and TABLE_NAME in ('{"', '".join(tbl_keys.keys())}')
order by a.TABLE_NAME, a.INHERITANCE_DEPTH, dict.ELEMENT
;
"""

In [ ]:
conn, _ = getSnowFlakeConn('LZ_DEV', 'SERVICENOW', [])
res = conn.cursor().execute(ancestors_sql)
data_types = res.fetch_pandas_all()

In [ ]:
data_types#[(data_types['TABLE_NAME'] == 'sc_task')].iat[0, 1]

In [ ]:
lz_template = """
create or replace table {sf_table} (
    _FILENAME VARCHAR(16777216),
    _FILEDATE TIMESTAMP_NTZ(9),
    _UPLOADDATE TIMESTAMP_NTZ(9),
    JSONFIELD VARIANT
);
"""

bronze_template = """
create or replace table SERVICENOW_{sf_table} (
    _AUDIT_LOG_ID VARCHAR(36),
    _ROW_HASH VARCHAR(128),
    _FILEDATE TIMESTAMP_NTZ(9),
    _REFRESHDATE TIMESTAMP_NTZ(9),
    SYS_ID VARCHAR(16777216),
    SYS_CREATED_ON TIMESTAMP_NTZ(9),
    SYS_CREATED_BY VARCHAR(16777216),
    SYS_UPDATED_ON TIMESTAMP_NTZ(9),
    SYS_UPDATED_BY VARCHAR(16777216),
    SYS_MOD_COUNT NUMBER(38,0),
    {sf_columns},
    primary key (SYS_ID)
);
"""


In [ ]:
dbt_template = """{{{{ config(
    alias = 'SERVICENOW_{sf_table}',
    materialized = 'tel_incremental',
    incremental_strategy = 'merge',
    unique_key = ['SYS_ID'],
) }}}}

with source_data as (
    select
        _FILEDATE,
        JSONFIELD:"sys_id"::VARCHAR(16777216) as SYS_ID,
        JSONFIELD:"sys_created_on"::string as SYS_CREATED_ON,
        JSONFIELD:"sys_created_by"::string as SYS_CREATED_BY,
        JSONFIELD:"sys_updated_on"::string as SYS_UPDATED_ON,
        JSONFIELD:"sys_updated_by"::string as SYS_UPDATED_BY,
        JSONFIELD:"sys_mod_count"::string as SYS_MOD_COUNT,        
        {src_data},
    from {{{{ source('SERVICENOW', '{sf_table}') }}}} 
    where _FILEDATE > ifnull((select max(_FILEDATE) from {{{{ this }}}}), '1970-01-01')
    qualify row_number() over (partition by SYS_ID order by _FILEDATE desc, _UPLOADDATE desc) = 1
),

transform_data as (
    select 
        '{{{{ invocation_id }}}}' AS _AUDIT_LOG_ID,
        SHA2('hash_me') AS _ROW_HASH,
        current_timestamp()::timestamp_ntz as _REFRESHDATE,
        *,
    from source_data
),

type_data as (
    select 
        _AUDIT_LOG_ID,
        _ROW_HASH,
        _FILEDATE,
        _REFRESHDATE,
        SYS_ID::VARCHAR(16777216) as SYS_ID,
        convert_timezone('UTC', 'America/Chicago', SYS_CREATED_ON) as SYS_CREATED_ON,
        SYS_CREATED_BY::VARCHAR(16777216) as SYS_CREATED_BY,
        convert_timezone('UTC', 'America/Chicago', SYS_UPDATED_ON) as SYS_UPDATED_ON,
        SYS_UPDATED_BY::VARCHAR(16777216) as SYS_UPDATED_BY,
        SYS_MOD_COUNT::NUMBER(38,0) as SYS_MOD_COUNT,
        {type_data},
    from transform_data
)

select * from type_data
"""

In [ ]:
dtype_mappings = {
    'string': 'VARCHAR(16777216)',
    'reference': 'VARCHAR(16777216)',
    'glide_date_time': 'TIMESTAMP_NTZ(9)',
    'integer': 'NUMBER(38,0)',
    'boolean': 'BOOLEAN',
    'glide_date': 'DATE',
    'choice': 'VARCHAR(16777216)',
    'float': 'NUMBER(38,10)',
    'decimal': 'NUMBER(38,10)',
    'json': 'OBJECT',
    'guid': 'VARCHAR(16777216)',
}


In [ ]:
bronze_snow = ''
lz_snow = ''

config_file = Path(r'C:\Sources\GitHub\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_ServiceNow.json')
config_data = json.loads(config_file.read_text(encoding="utf-8"))

src_path = Path(r'C:\Sources\GitHub\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\_src\servicenow_sources.yml')
src_yml = yaml.safe_load(src_path.read_text(encoding="utf-8"))
src_tables = src_yml['sources'][0]['tables']

models_path = Path(r'C:\Sources\GitHub\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\teh_support_services\bronze\_models.yml')
models_yml = yaml.safe_load(models_path.read_text(encoding="utf-8"))
model_entries = models_yml['models']


top_keys = ['sys_id', 'sys_created_on', 'sys_created_by', 'sys_updated_on', 'sys_updated_by', 'sys_mod_count']
reserved_keywords = {
    'unique': 'IS_UNIQUE',
    'order': 'ORDER_NUMBER',
    'start': 'START_DATETIME',
    'end': 'END_DATETIME',
    'group': 'GROUP_ID',
    'table': 'TABLE_NAME',
}

for table_name, keys in tbl_keys.items():
    
    # if table_name != 'metric_instance':
    #     continue

    config_data['resources'][table_name] = {
        "table_name": table_name,
        "query": "",
        "fields": "",
        "sf_table": table_name.upper()
    }

    type_data = []
    src_data = []
    bronze_cols = []

    old_src = None
    new_src = {
        "name": table_name.upper(),
        "identifier": table_name.upper(),
        "description": f"Landing table for {table_name.upper()} file."
    }
    for src in src_tables:
        if src['name'].upper() == table_name.upper():
            old_src = src
    if old_src:
        old_src.update(new_src)
    else:
        src_tables.append(new_src)
    
    tbl_desc = data_types[(data_types['TABLE_NAME'] == table_name)].iat[0, 1]
    yml_cols = [
        {'name': '_AUDIT_LOG_ID', 'description': 'Unique run identifier for the dbt invocation that processed this row.'},
        {'name': '_ROW_HASH', 'description': 'Deterministic hash of row content used for change tracking.'},
        {'name': '_FILEDATE', 'description': 'Ingestion file timestamp from the landing/source file metadata.'},
        {'name': '_REFRESHDATE', 'description': 'Timestamp when this row was transformed/refreshed in Bronze.'},
        {'name': 'SYS_ID', 'description': 'Unique ServiceNow record identifier (primary key).'},
        {'name': 'SYS_CREATED_ON', 'description': 'UTC timestamp when the record was created in ServiceNow.'},
        {'name': 'SYS_CREATED_BY', 'description': 'ServiceNow user who created the record.'},
        {'name': 'SYS_UPDATED_ON', 'description': 'UTC timestamp when the record was last updated in ServiceNow.'},
        {'name': 'SYS_UPDATED_BY', 'description': 'ServiceNow user who last updated the record.'},
        {'name': 'SYS_MOD_COUNT', 'description': 'Number of updates made to the ServiceNow record.'},
    ]
    new_mdl = {
        "name": f"BRONZE__TEH_SUPPORT_SERVICES__SERVICENOW_{table_name.upper()}",
        "description": tbl_desc,
        "tests": ["has_data", "recently_run"],
        "columns": yml_cols
    }

    
    for k in keys:
        if k not in top_keys:
            column_name = reserved_keywords.get(k, k.upper())
            element_matches = data_types[(data_types['TABLE_NAME'] == table_name) & (data_types['COLUMN_NAME'] == k)]
            if element_matches.empty or pd.isna(element_matches.iat[0, 3]):
                sn_dtype = 'string'  # fallback when element is not found
                col_desc = None
            else:
                sn_dtype = element_matches.iat[0, 5]
                col_desc = element_matches.iat[0, 3]
            yml_cols += [
                {
                    "name": k.upper(),
                    "description": col_desc,
                }
            ]
            sf_dtype = dtype_mappings.get(sn_dtype, 'VARCHAR(16777216)')
            src_data.append(f'JSONFIELD:"{k}"::string as {column_name}')
            if sf_dtype == 'TIMESTAMP_NTZ(9)':
                type_data.append(f"convert_timezone('UTC', 'America/Chicago', {column_name}) as {column_name}")
            else:
                type_data.append(f'{column_name}::{sf_dtype} as {column_name}')
            bronze_cols.append(f'{column_name} {sf_dtype}')

            if table_name == 'metric_definition' and k == 'active':
                print(f"{sf_dtype=}, ")
    
    lz_snow += lz_template.format(sf_table=table_name.upper())
    bronze_snow += bronze_template.format(sf_table=table_name.upper(), sf_columns=',\n    '.join(bronze_cols))

    new_mdl['columns'] = yml_cols
    old_mdl = None
    for model_entry in model_entries:
        if model_entry['name'] == f"BRONZE__TEH_SUPPORT_SERVICES__SERVICENOW_{table_name.upper()}":
            old_mdl = model_entry
            break
    if old_mdl:
        old_mdl.update(model_entry)
    else:
        model_entries.append(model_entry)
    model_entries.sort(key=lambda x: x['name'])
    models_yml['models'] = model_entries

    src_data = ',\n        '.join(src_data)
    type_data = ',\n        '.join(type_data)
    dbt_model = dbt_template.format(sf_table=table_name.upper(), src_data=src_data, type_data=type_data)

    folder_path = r"C:\Sources\\GitHub\\EnterpriseCloud\\astro_datautility\\dags\\dbt\\dbt_tel\\models\teh_support_services\\bronze\\"
    dbt_file = Path(folder_path + f"BRONZE__TEH_SUPPORT_SERVICES__SERVICENOW_{ table_name.upper() }.SQL")
    dbt_file.write_text(dbt_model, encoding="utf-8")

sn_snow = f"""use schema <<LZ>>.SERVICENOW;
{lz_snow}

use schema <<BRONZE>>.TEH_SUPPORT_SERVICES;
{bronze_snow}
"""
sn_snow_file = Path("C:\\Sources\\GitHub\\EnterpriseCloud\\Snow\\_sn.sql")
sn_snow_file.write_text(sn_snow, encoding="utf-8")

config_data['resources'] = dict(sorted(config_data['resources'].items()))
config_file.write_text(json.dumps(config_data, indent=4), encoding="utf-8")

src_tables.sort(key=lambda x: x['name'])
with src_path.open('w', encoding='utf-8') as f:
    yaml.dump(
        src_yml,
        f,
        Dumper=IndentSafeDumper,
        sort_keys=False,
        allow_unicode=True,
        indent=2,
        default_flow_style=False,
    )

with models_path.open('w', encoding='utf-8') as f:
    yaml.dump(
        models_yml,
        f,
        Dumper=IndentSafeDumper,
        sort_keys=False,
        allow_unicode=True,
        indent=2,
        default_flow_style=False,
    )